# Day 4: Baseline Model Training

Day 4 trains baseline fraud detection models on the Day 3 processed data.

- The goal is to establish a benchmark before moving on to advanced models (XGBoost/LightGBM) in Day 5.
- Only the validation set is used for evaluation in this notebook.
- The test set remains completely untouched and is reserved for final evaluation later in the project.

## Day 4 Objectives

- Load Day 3 processed train and validation data.
- Train a Dummy baseline model.
- Train a Logistic Regression baseline model.
- Train a Random Forest baseline model.
- Evaluate all baselines using fraud-aware metrics.
- Compare baselines using PR-AUC, recall, precision, and F1-score.
- Save model artifacts and metrics through the Day 4 script (`scripts/run_day4_baseline_models.py`), not from this notebook.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Add the project root to sys.path so this notebook works both when run
# from the notebooks/ folder and from the repository root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.baseline_models import train_baseline_models, get_model_summary
from src.evaluation.classification_metrics import (
    evaluate_models,
    metrics_to_dataframe,
    find_best_model_by_pr_auc,
)

In [ ]:
PROCESSED_DIR = Path("../data/processed")
if not PROCESSED_DIR.exists():
    PROCESSED_DIR = Path("data/processed")

PROCESSED_DIR

## 1. Load Processed Train and Validation Data

In [ ]:
X_train = pd.read_parquet(PROCESSED_DIR / "X_train_processed.parquet")
X_val = pd.read_parquet(PROCESSED_DIR / "X_val_processed.parquet")
y_train = pd.read_parquet(PROCESSED_DIR / "y_train.parquet")["Class"]
y_val = pd.read_parquet(PROCESSED_DIR / "y_val.parquet")["Class"]

print("X_train shape:", X_train.shape)
print("X_val shape:  ", X_val.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:  ", y_val.shape)

## 2. Confirm Test Set Is Not Used

The test set should remain untouched until the project's final evaluation stage. Day 4 deliberately uses the validation set only, both for comparing baselines and for evaluating model performance.

In [ ]:
print("X_train.shape:", X_train.shape)
print("X_val.shape:  ", X_val.shape)

print("\ny_train class distribution:")
print(y_train.value_counts())

print("\ny_val class distribution:")
print(y_val.value_counts())

print("\nIs 'Class' present in X_train columns?", "Class" in X_train.columns)
print("Is 'Class' present in X_val columns?  ", "Class" in X_val.columns)

## 3. Train Baseline Models

Three baseline models are trained on the training data:

- **Dummy baseline** -- shows naive majority-class performance and demonstrates why accuracy alone is misleading on imbalanced data.
- **Logistic Regression** -- a simple, interpretable linear baseline.
- **Random Forest** -- a non-linear, tree-based baseline.

Logistic Regression and Random Forest use balanced class weights to help address the severe class imbalance between legitimate and fraudulent transactions.

In [ ]:
models = train_baseline_models(X_train, y_train)
model_summary = get_model_summary(models)
model_summary

## 4. Evaluate on Validation Set

Accuracy alone is not enough for this problem, since fraud is a rare positive class. Precision, recall, F1-score, ROC-AUC, PR-AUC, and the confusion matrix give a much more complete picture of how well each baseline detects fraud.

In [ ]:
all_metrics = evaluate_models(models, X_val, y_val)
metrics_df = metrics_to_dataframe(all_metrics)
metrics_df

## 5. Best Baseline by PR-AUC

PR-AUC is especially useful when the positive class (fraud) is rare, since it focuses on the precision-recall tradeoff rather than overall accuracy. A higher PR-AUC indicates a better balance between catching fraud and avoiding false alarms.

In [ ]:
best_model = find_best_model_by_pr_auc(all_metrics)
best_model

## 6. Confusion Matrix Interpretation

- **True negatives** -- legitimate transactions correctly classified as legitimate.
- **False positives** -- legitimate transactions incorrectly flagged as fraud.
- **False negatives** -- fraudulent transactions the model missed.
- **True positives** -- fraudulent transactions the model correctly caught.

False negatives are especially costly in fraud detection, since a missed fraud case translates directly into financial loss.

In [ ]:
confusion_columns = [
    "model_name",
    "true_negatives",
    "false_positives",
    "false_negatives",
    "true_positives",
]
metrics_df[confusion_columns]

## 7. Key Day 4 Findings

- Baseline models (Dummy, Logistic Regression, Random Forest) were trained on the Day 3 processed training data.
- Validation metrics were generated for every baseline.
- The Dummy baseline illustrates why accuracy alone is misleading under severe class imbalance.
- Logistic Regression and Random Forest establish meaningful benchmarks for future, more advanced models.
- The test set was not used anywhere in this notebook and remains untouched.

## 8. Day 5 Next Steps

- Train an XGBoost or LightGBM model.
- Compare the advanced model against the Day 4 baselines.
- Continue prioritizing PR-AUC, recall, precision, and F1-score over raw accuracy.
- Avoid selecting a final model based on accuracy alone.